# TRƯỜNG ĐẠI HỌC CÔNG NGHỆ THÔNG TIN - ĐHQG-HCM
### KHOA KHOA HỌC DỮ LIỆU & TRÍ TUỆ NHÂN TẠO
---
## BÁO CÁO BÀI TẬP THỰC HÀNH: LAB 1 - THU THẬP DỮ LIỆU WEB (WEB SCRAPING)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/barackvn/MayHocNangCao/blob/main/Buoi2-KhaiPhaVaXulyDuLieu/Lab1_ThuThapDuLieu_WebScraping.ipynb)

* **Môn học:** DS200 - Phân tích dữ liệu lớn (Big Data Analytics)
* **Lớp:** DS200.G11.CN2.TTNT
* **Giảng viên hướng dẫn:** ThS. Hà Minh Tân
* **Học viên / Sinh viên thực hiện:** barackvn
* **Email:** hotro.76@gmail.com
* **Mục tiêu bài lab:** Khai phá & thu thập dữ liệu tự động từ các nền tảng website phục vụ phân tích dữ liệu lớn.

## I. MỤC TIÊU VÀ YÊU CẦU BÀI LAB
1. **Nắm vững cơ chế HTTP/HTTPS:** Hiểu cách gửi HTTP Request và nhận Response từ máy chủ web bằng thư viện `requests`.
2. **Khai phá cấu trúc cây DOM (HTML Parsing):** Sử dụng thư viện `BeautifulSoup4` để duyệt cây DOM, tìm kiếm theo thẻ (`tag`), tên lớp (`class`), và trích xuất thuộc tính (`attributes`).
3. **Thu thập danh sách bài báo:** Crawl dữ liệu tin tức thể thao/bóng đá từ VnExpress (`https://vnexpress.net/bong-da`).
4. **Trích xuất thông tin chi tiết từng bài:** Bóc tách tiêu đề (`title`), tóm tắt (`abstract`/`description`), thời gian đăng, nội dung chi tiết.
5. **Yêu cầu 5:** Tổ chức dữ liệu bài viết vào cấu trúc bảng (Pandas DataFrame) và lưu trữ thành file `data_vnexpress_bongda.csv` chuẩn UTF-8 (mở trên Excel tiếng Việt không lỗi font).
6. **Yêu cầu 6:** Mở rộng thu thập dữ liệu từ trang báo khác (Báo Dân Trí `dantri.com.vn`), so sánh cấu trúc DOM và xuất file CSV tương ứng.

---
## II. THỰC HÀNH
### 1. Cài đặt và nạp các thư viện cần thiết
Các thư viện sử dụng:
* `requests`: Gửi HTTP requests để tải mã nguồn HTML của trang web.
* `beautifulsoup4`: Phân tích cú pháp và bóc tách dữ liệu từ cây HTML DOM.
* `pillow`: Xử lý hình ảnh (theo đề cương bài thực hành).
* `pandas`: Tổ chức dữ liệu dạng bảng và xuất file CSV.

In [ ]:
# Cài đặt các thư viện cần thiết theo yêu cầu bài Lab
!pip install -q requests beautifulsoup4 pillow pandas lxml

In [ ]:
# Nạp các thư viện vào môi trường làm việc
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
from datetime import datetime

print("✅ Đã nạp thành công các thư viện!")
print(f"- Requests version: {requests.__version__}")
print(f"- BeautifulSoup4: bs4")
print(f"- Pandas version: {pd.__version__}")

---
### 2. Gửi HTTP Request tới trang web đích
* URL mục tiêu: `https://vnexpress.net/bong-da`
* Sử dụng `requests.get()` cùng `User-Agent` header chuẩn để giả lập trình duyệt, đảm bảo kết nối ổn định và nhận mã trạng thái `200 OK`.

In [ ]:
# Thiết lập URL và User-Agent
url = "https://vnexpress.net/bong-da"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

# Gửi yêu cầu GET tới máy chủ VnExpress
response = requests.get(url, headers=headers)

# Kiểm tra phản hồi từ máy chủ
print("Kết quả HTTP Response:", response)
print("Mã trạng thái HTTP (Status Code):", response.status_code)
print("Định dạng mã hóa (Encoding):", response.encoding)

In [ ]:
# Xem thử 500 ký tự đầu tiên của mã HTML trả về
print("--- 500 ký tự đầu tiên của mã HTML ---")
print(response.text[:500])

---
### 3. Tách và phân tích dữ liệu HTML với BeautifulSoup4
* `BeautifulSoup` chuyển đổi mã HTML thô thành cây đối tượng có phân cấp (DOM Tree).
* Cho phép tìm kiếm thẻ theo tên (`tag`), lớp (`class`), `id`, hoặc biểu thức CSS selector.

In [ ]:
# Khởi tạo đối tượng BeautifulSoup với parser 'html.parser'
soup = BeautifulSoup(response.content, "html.parser")

# Trích xuất tiêu đề của trang web (thẻ <title>)
print("Tiêu đề trang web (Page Title):", soup.title.string if soup.title else "None")

---
### 4. Phân tích cấu trúc và lấy danh sách liên kết bài viết
* Trên trang VnExpress, các tiêu đề bài viết nằm trong thẻ `<h2>` hoặc `<h3>` có `class="title-news"`.
* Bên trong chứa thẻ `<a>` với thuộc tính `href` lưu đường dẫn URL đến trang chi tiết của bài báo.

In [ ]:
# Tìm tất cả các thẻ tiêu đề bài viết
titles = soup.find_all(["h2", "h3"], class_="title-news")
print(f"Tổng số tiêu đề tìm thấy trên trang: {len(titles)}")

# Trích xuất danh sách liên kết bài báo hợp lệ
links = []
for item in titles:
    a_tag = item.find("a")
    if a_tag and "href" in a_tag.attrs:
        href = a_tag["href"]
        # Lọc các liên kết thuộc vnexpress và có đuôi .html
        if href.startswith("https://vnexpress.net") and href.endswith(".html"):
            if href not in links:
                links.append(href)

print(f"✅ Đã lọc được {len(links)} liên kết bài viết duy nhất.")
print("\n--- Danh sách 5 liên kết đầu tiên ---")
for i, link in enumerate(links[:5], start=1):
    print(f"{i}. {link}")

---
### 5. Thu thập thông tin chi tiết từng bài viết
Truy cập vào từng liên kết trong danh sách để trích xuất các trường thông tin:
* **Tiêu đề (`title`):** Thẻ `<h1 class="title-detail">`
* **Tóm tắt (`abstract`):** Thẻ `<p class="description">`
* **Thời gian đăng (`publish_date`):** Thẻ `<span class="date">`
* **Nội dung bài viết (`content`):** Các thẻ `<p class="Normal">` trong thẻ `<article class="fck_detail">`

> **Lưu ý kỹ thuật:** Thiết lập độ trễ `time.sleep(0.3)` giữa các request nhằm tuân thủ quy tắc thu thập dữ liệu lịch sự (Polite Web Scraping), không gây nghẽn máy chủ.

In [ ]:
# Danh sách chứa dữ liệu thu thập
articles_data = []

# Chọn lấy mẫu 15 bài đầu tiên
num_articles = 15
target_links = links[:num_articles]

print(f"Bắt đầu thu thập dữ liệu chi tiết cho {len(target_links)} bài viết...\n")

for idx, link in enumerate(target_links, start=1):
    try:
        news_resp = requests.get(link, headers=headers, timeout=10)
        if news_resp.status_code != 200:
            continue
            
        news_soup = BeautifulSoup(news_resp.content, "html.parser")
        
        # 1. Tiêu đề bài viết
        title_tag = news_soup.find("h1", class_="title-detail") or news_soup.find("h1")
        title = title_tag.get_text(strip=True) if title_tag else ""
        
        # 2. Tóm tắt bài viết
        desc_tag = news_soup.find("p", class_="description")
        if desc_tag:
            abstract = desc_tag.get_text(strip=True)
        else:
            meta_desc = news_soup.find("meta", {"name": "description"})
            abstract = meta_desc.get("content", "").strip() if meta_desc else ""
            
        # 3. Ngày giờ xuất bản
        date_tag = news_soup.find("span", class_="date")
        publish_date = date_tag.get_text(strip=True) if date_tag else ""
        
        # 4. Nội dung bài viết
        paragraphs = news_soup.select("article.fck_detail p.Normal")
        content = " ".join([p.get_text(strip=True) for p in paragraphs if p.get_text(strip=True)])
        
        # Lưu vào danh sách
        articles_data.append({
            "STT": idx,
            "TieuDe": title,
            "MoTa": abstract,
            "NgayDang": publish_date,
            "NoiDung": content,
            "DuongDan": link,
            "ThoiGianCrawl": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        })
        
        print(f"[{idx:02d}/{len(target_links)}] {title[:70]}...")
        time.sleep(0.3)
        
    except Exception as e:
        print(f"[{idx:02d}] Lỗi khi tải {link}: {e}")

print(f"\n✅ Đã thu thập thành công {len(articles_data)} bài báo chi tiết.")

---
### 6. Yêu cầu 5: Lưu trữ dữ liệu thành tập tin CSV
* Chuyển danh sách dữ liệu sang `pandas.DataFrame`.
* Xuất tập tin `data_vnexpress_bongda.csv` với định dạng mã hóa `utf-8-sig` (hỗ trợ hiển thị đúng tiếng Việt có dấu trong Microsoft Excel).

In [ ]:
# Khởi tạo DataFrame từ dữ liệu đã crawl
df_articles = pd.DataFrame(articles_data)

# Hiển thị 5 bản ghi đầu tiên
print("Bảng dữ liệu trích xuất (5 dòng đầu):")
df_articles[["STT", "TieuDe", "NgayDang", "MoTa"]].head()

In [ ]:
# Thống kê tổng quan cấu trúc DataFrame
df_articles.info()

In [ ]:
# Lưu trữ vào tệp tin CSV
csv_filename = "data_vnexpress_bongda.csv"
df_articles.to_csv(csv_filename, index=False, encoding="utf-8-sig")

print(f"💾 Đã lưu thành công dữ liệu vào tệp: {csv_filename}")
print(f"- Số lượng bản ghi: {len(df_articles)}")
print(f"- Số lượng thuộc tính (cột): {df_articles.shape[1]}")

In [ ]:
# Tải file CSV về máy tính cá nhân (khi chạy trên Google Colab)
try:
    from google.colab import files
    files.download(csv_filename)
    print("⬇️ Đã kích hoạt tải tập tin CSV về máy tính.")
except ImportError:
    print("Môi trường cục bộ: Tệp tin đã được lưu tại thư mục hiện hành.")

---
### 7. Yêu cầu 6: Mở rộng thu thập dữ liệu từ Website tin tức khác
* Áp dụng quy trình thu thập dữ liệu trên trang tin **Báo Dân Trí (`dantri.com.vn`)** chuyên mục Thể thao.
* So sánh cấu trúc bóc tách dữ liệu DOM giữa các trang web.

In [ ]:
# Hàm thu thập tin tức từ Báo Dân Trí
def crawl_dantri(limit=10):
    dantri_url = "https://dantri.com.vn/the-thao.htm"
    headers = {"User-Agent": "Mozilla/5.0"}
    res = requests.get(dantri_url, headers=headers, timeout=10)
    soup = BeautifulSoup(res.content, "html.parser")
    
    # Bóc tách danh sách bài viết trên Dân Trí
    links = []
    for a in soup.select("h3.article-title a"):
        href = a.get("href", "")
        if href.endswith(".htm"):
            full = "https://dantri.com.vn" + href if href.startswith("/") else href
            if full not in links:
                links.append(full)
                
    print(f"Tìm thấy {len(links)} bài báo trên Dân Trí. Bắt đầu thu thập {limit} bài...")
    
    results = []
    for i, link in enumerate(links[:limit], start=1):
        try:
            r = requests.get(link, headers=headers, timeout=10)
            sp = BeautifulSoup(r.content, "html.parser")
            
            title = sp.find("h1").get_text(strip=True) if sp.find("h1") else ""
            desc = sp.find("h2").get_text(strip=True) if sp.find("h2") else ""
            
            results.append({
                "STT": i,
                "Nguon": "Dân Trí",
                "TieuDe": title,
                "MoTa": desc,
                "DuongDan": link,
                "ThoiGianCrawl": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            })
            print(f"[{i:02d}] {title[:70]}...")
            time.sleep(0.3)
        except Exception as err:
            print(f"[{i:02d}] Lỗi: {err}")
            
    return pd.DataFrame(results)

# Thực thi thu thập từ Dân Trí
df_dantri = crawl_dantri(limit=5)
df_dantri.to_csv("data_dantri_thethao.csv", index=False, encoding="utf-8-sig")
print("\n💾 Đã lưu dữ liệu Dân Trí vào file: data_dantri_thethao.csv")
df_dantri.head()

---
## III. KẾT LUẬN VÀ NHẬN XÉT
1. **Khả năng ứng dụng:** Thư viện `requests` kết hợp cùng `BeautifulSoup4` mang lại giải pháp thu thập dữ liệu web tĩnh nhanh chóng, hiệu quả và dễ tùy biến cho các bài toán phân tích dữ liệu lớn.
2. **Sự đa dạng của cấu trúc DOM:** Mỗi trang báo (VnExpress, Dân Trí, Tuổi Trẻ...) sử dụng các thẻ HTML và quy tắc đặt tên `class`/`id` khác nhau. Khi xây dựng crawler cần khảo sát kỹ cấu trúc mã nguồn bằng công cụ F12 (Inspect Element) và thiết kế cơ chế xử lý ngoại lệ an toàn.
3. **Chuẩn hóa dữ liệu:** Để phục vụ các bước tiền xử lý trong học phần Phân tích dữ liệu lớn, việc lưu trữ với bảng mã `utf-8-sig` là tiêu chuẩn bắt buộc khi làm việc với tiếng Việt.